In [ ]:
#|default_exp datasource

In [ ]:
#|hide
%load_ext autoreload
%autoreload 2

## DataSource

This notebook creates an abstract class that will be used to standardize the different data sources.

In [ ]:
#| export

from abc import ABC, abstractmethod
from pathlib import Path
from typing import List, Dict, Any
import polars as pl
from pydantic import BaseModel, Field
import logging

In [ ]:
#| export

logging.basicConfig(level=logging.INFO)

In [ ]:
#| export

class DataSource(ABC):
    """An abstract base class for data sources."""

    def __init__(self, data_folder: Path):
        self.data_folder = data_folder
        self.data_folder.mkdir(parents=True, exist_ok=True)

    @abstractmethod
    def get_tokens(self) -> List[str]:
        """Returns a list of available tokens."""
        pass

    @abstractmethod
    def get_perp_prices(self, token: str, start_date: str, end_date: str, time_interval: str) -> pl.DataFrame:
        """Returns a DataFrame of perpetual prices for a given token."""
        pass

    @abstractmethod
    def get_spot_prices(self, token: str, start_date: str, end_date: str, time_interval: str) -> pl.DataFrame:
        """Returns a DataFrame of spot prices for a given token."""
        pass

    @abstractmethod
    def get_funding_rates(self, token: str, start_date: str, end_date: str) -> pl.DataFrame:
        """Returns a DataFrame of funding rates for a given token."""
        pass

    def save_data(self, df: pl.DataFrame, token: str, file_type: str = 'parquet'):
        """Saves the data to a file."""
        file_path = self.data_folder / f"{token}.{file_type}"
        if file_type == 'parquet':
            df.write_parquet(file_path)
        elif file_type == 'csv':
            df.write_csv(file_path)
        else:
            raise ValueError(f"Unsupported file type: {file_type}")
            
    def read_data(self, token: str, file_type: str = 'parquet') -> pl.DataFrame:
        """Reads the data from a file."""
        file_path = self.data_folder / f"{token}.{file_type}"
        if not file_path.exists():
            return pl.DataFrame()
        if file_type == 'parquet':
            return pl.read_parquet(file_path)
        elif file_type == 'csv':
            return pl.read_csv(file_path)
        else:
            raise ValueError(f"Unsupported file type: {file_type}")
